## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, EXTERNAL_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ External directory: {EXTERNAL_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

## 1. Cargar Dataset del Paso Anterior (Step 4 - Climate Features)

In [ ]:
# Cargar dataset con climate features del paso anterior
df_features = pd.read_csv(PROCESSED_DIR / 'features_step4_climate.csv', index_col=0, parse_dates=True)

print("=" * 80)
print("DATASET CARGADO (Step 4 - Climate Features)")
print("=" * 80)
print(f"Shape: {df_features.shape}")
print(f"Período: {df_features.index.min()} → {df_features.index.max()}")
print(f"Features actuales: {len(df_features.columns):,}")
print(f"\nÚltimas 10 columnas: {df_features.columns[-10:].tolist()}")

## 2. Cargar CFTC COT Data (Soybeans)

In [ ]:
# Cargar datos CFTC diarios (ya expandidos de semanal a diario)
cftc_file = EXTERNAL_DIR / 'cftc' / 'cftc_soybeans_daily.csv'

if not cftc_file.exists():
    print(f"❌ ERROR: Archivo CFTC no encontrado: {cftc_file}")
    print("   Ejecutar primero: python src/data/download_cftc_cot.py")
    raise FileNotFoundError(f"CFTC data not found at {cftc_file}")

df_cftc = pd.read_csv(cftc_file)
df_cftc['date'] = pd.to_datetime(df_cftc['date'])
df_cftc = df_cftc.set_index('date')

# Renombrar columnas agregando prefijo cftc_ para claridad
rename_cols = {
    'open_interest': 'cftc_open_interest',
    'managed_long': 'cftc_managed_long',
    'managed_short': 'cftc_managed_short',
    'managed_net': 'cftc_managed_net',
    'managed_net_pct': 'cftc_managed_net_pct',
    'producer_long': 'cftc_producer_long',
    'producer_short': 'cftc_producer_short',
    'producer_net': 'cftc_producer_net',
    'producer_net_pct': 'cftc_producer_net_pct',
    'swap_long': 'cftc_swap_long',
    'swap_short': 'cftc_swap_short',
    'swap_net': 'cftc_swap_net',
    'swap_net_pct': 'cftc_swap_net_pct',
    'other_long': 'cftc_other_long',
    'other_short': 'cftc_other_short',
    'other_net': 'cftc_other_net'
}
df_cftc = df_cftc.rename(columns=rename_cols)

print("=" * 80)
print("CFTC COMMITMENTS OF TRADERS DATA")
print("=" * 80)
print(f"Shape: {df_cftc.shape}")
print(f"Período: {df_cftc.index.min()} → {df_cftc.index.max()}")
print(f"Días con datos: {len(df_cftc):,}")
print(f"\nColumnas CFTC (después de renombrar):")
for col in df_cftc.columns:
    print(f"  - {col}")

print(f"\nEstadísticas de posiciones:")
print(df_cftc[['cftc_managed_net', 'cftc_producer_net', 'cftc_managed_net_pct', 'cftc_open_interest']].describe())

## 3. Feature Engineering: Momentum y Cambios

Calcular cambios week-over-week y momentum de posiciones especulativas.

In [ ]:
# Calcular cambios semanales (7 días = 1 semana)
df_cftc['cftc_managed_net_change_7d'] = df_cftc['cftc_managed_net'].diff(7)
df_cftc['cftc_managed_net_pct_change_7d'] = df_cftc['cftc_managed_net_pct'].diff(7)
df_cftc['cftc_open_interest_change_7d'] = df_cftc['cftc_open_interest'].diff(7)

# Calcular ratios (managed vs producer)
df_cftc['cftc_managed_producer_ratio'] = df_cftc['cftc_managed_net'] / df_cftc['cftc_producer_net'].replace(0, np.nan)

# Calcular percentiles históricos (rolling 252 días = 1 año trading)
df_cftc['cftc_managed_net_percentile_252d'] = (
    df_cftc['cftc_managed_net']
    .rolling(window=252, min_periods=50)
    .apply(lambda x: (x.iloc[-1] <= x).sum() / len(x) * 100)
)

# Señal de extremos (sentiment extremo puede indicar reversión)
# Percentil > 90 = posición extremadamente larga (potential reversal)
# Percentil < 10 = posición extremadamente corta (potential reversal)
df_cftc['cftc_extreme_long'] = (df_cftc['cftc_managed_net_percentile_252d'] > 90).astype(int)
df_cftc['cftc_extreme_short'] = (df_cftc['cftc_managed_net_percentile_252d'] < 10).astype(int)

print("=" * 80)
print("NUEVAS FEATURES GENERADAS")
print("=" * 80)
print("1. Momentum/Cambios:")
print("   - cftc_managed_net_change_7d: Cambio semanal en contratos")
print("   - cftc_managed_net_pct_change_7d: Cambio semanal en %")
print("   - cftc_open_interest_change_7d: Cambio en Open Interest")
print("\n2. Ratios:")
print("   - cftc_managed_producer_ratio: Especuladores / Hedgers")
print("\n3. Percentiles y extremos:")
print("   - cftc_managed_net_percentile_252d: Percentil vs último año")
print("   - cftc_extreme_long: 1 si percentil > 90 (potencial reversión)")
print("   - cftc_extreme_short: 1 si percentil < 10 (potencial reversión)")

# Mostrar estadísticas
print(f"\nEstadísticas de nuevas features:")
new_features = [
    'cftc_managed_net_change_7d', 
    'cftc_managed_net_pct_change_7d',
    'cftc_managed_producer_ratio',
    'cftc_managed_net_percentile_252d'
]
print(df_cftc[new_features].describe())

# Contar extremos
print(f"\nExtreme positions:")
print(f"  - Extreme long: {df_cftc['cftc_extreme_long'].sum()} días ({df_cftc['cftc_extreme_long'].mean()*100:.1f}%)")
print(f"  - Extreme short: {df_cftc['cftc_extreme_short'].sum()} días ({df_cftc['cftc_extreme_short'].mean()*100:.1f}%)")

## 4. Merge con Dataset Principal

Unir CFTC features con el dataset existente.

In [ ]:
# Verificar overlap temporal
print("=" * 80)
print("VERIFICACIÓN DE OVERLAP TEMPORAL")
print("=" * 80)
print(f"Dataset principal: {df_features.index.min()} → {df_features.index.max()} ({len(df_features)} días)")
print(f"CFTC data:         {df_cftc.index.min()} → {df_cftc.index.max()} ({len(df_cftc)} días)")

# Identificar overlap
overlap_start = max(df_features.index.min(), df_cftc.index.min())
overlap_end = min(df_features.index.max(), df_cftc.index.max())
print(f"\nOverlap period:    {overlap_start} → {overlap_end}")

# Merge left join (mantener todas las fechas del dataset principal)
df_merged = df_features.merge(
    df_cftc, 
    left_index=True, 
    right_index=True, 
    how='left',
    suffixes=('', '_cftc_dup')
)

print(f"\nShape después del merge: {df_merged.shape}")
print(f"Features agregadas: {df_merged.shape[1] - df_features.shape[1]}")

# Verificar NaNs introducidos por CFTC data
cftc_cols = [col for col in df_cftc.columns]
missing_cftc = df_merged[cftc_cols].isnull().sum()
missing_cftc = missing_cftc[missing_cftc > 0].sort_values(ascending=False)

if len(missing_cftc) > 0:
    print(f"\n⚠️  Missing values en CFTC features:")
    print(missing_cftc)
    print(f"\nMotivo: CFTC data disponible desde {df_cftc.index.min()}")
    print(f"         Dataset principal empieza en {df_features.index.min()}")
else:
    print("\n✓ No hay missing values en CFTC features")

## 5. Visualización: Sentiment vs Precio Soybeans

Explorar relación entre posicionamiento especulativo y precio de soja.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# Crear figura con subplots
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

# Filtrar período con CFTC data
df_plot = df_merged.loc[df_cftc.index.min():].copy()

# 1. Precio Soybeans vs Managed Net Position
ax1 = axes[0]
ax1_twin = ax1.twinx()

ax1.plot(df_plot.index, df_plot['Soybeans'], color='green', linewidth=1.5, label='Soybeans Price')
ax1_twin.bar(df_plot.index, df_plot['cftc_managed_net'], color='orange', alpha=0.3, label='Managed Net Position')

ax1.set_ylabel('Soybeans Price (USD/bushel)', fontsize=11, color='green')
ax1_twin.set_ylabel('Managed Net Contracts', fontsize=11, color='orange')
ax1.set_title('Soybeans Price vs CFTC Managed Money Net Position', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# 2. Managed Net Percentage (normalizado)
ax2 = axes[1]
ax2_twin = ax2.twinx()

ax2.plot(df_plot.index, df_plot['Soybeans'], color='green', linewidth=1.5, label='Soybeans Price')
ax2_twin.plot(df_plot.index, df_plot['cftc_managed_net_pct'], color='blue', linewidth=1.5, label='Managed Net %')
ax2_twin.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

ax2.set_ylabel('Soybeans Price (USD/bushel)', fontsize=11, color='green')
ax2_twin.set_ylabel('Managed Net (% of Open Interest)', fontsize=11, color='blue')
ax2.set_title('Soybeans Price vs Managed Net Position (Normalized)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

# 3. Sentiment Extremes (potential reversals)
ax3 = axes[2]
ax3_twin = ax3.twinx()

ax3.plot(df_plot.index, df_plot['Soybeans'], color='green', linewidth=1.5, label='Soybeans Price')

# Marcar extremos
extreme_long_dates = df_plot[df_plot['cftc_extreme_long'] == 1].index
extreme_short_dates = df_plot[df_plot['cftc_extreme_short'] == 1].index

ax3.scatter(extreme_long_dates, df_plot.loc[extreme_long_dates, 'Soybeans'], 
            color='red', marker='v', s=50, alpha=0.7, label='Extreme Long (>90th percentile)')
ax3.scatter(extreme_short_dates, df_plot.loc[extreme_short_dates, 'Soybeans'], 
            color='blue', marker='^', s=50, alpha=0.7, label='Extreme Short (<10th percentile)')

ax3.set_ylabel('Soybeans Price (USD/bushel)', fontsize=11)
ax3.set_title('Soybeans Price with Extreme Sentiment Signals', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best')

# 4. Open Interest (liquidez del mercado)
ax4 = axes[3]
ax4_twin = ax4.twinx()

ax4.plot(df_plot.index, df_plot['Soybeans'], color='green', linewidth=1.5, label='Soybeans Price')
ax4_twin.plot(df_plot.index, df_plot['cftc_open_interest'], color='purple', linewidth=1.5, label='Open Interest')

ax4.set_ylabel('Soybeans Price (USD/bushel)', fontsize=11, color='green')
ax4_twin.set_ylabel('Open Interest (Contracts)', fontsize=11, color='purple')
ax4.set_title('Soybeans Price vs Market Liquidity (Open Interest)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Date', fontsize=11)
ax4.grid(True, alpha=0.3)
ax4.legend(loc='upper left')
ax4_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'cftc_sentiment_vs_soybeans.png', dpi=300, bbox_inches='tight')
print("\n✓ Gráfico guardado: reports/figures/cftc_sentiment_vs_soybeans.png")
plt.show()

## 6. Análisis de Correlación: CFTC Features vs Soybeans Returns

Verificar si sentiment tiene poder predictivo para retornos futuros.

In [ ]:
# Calcular retornos futuros de Soybeans (horizonte 7 días)
df_merged['Soybeans_return_7d_fwd'] = df_merged['Soybeans'].pct_change(7).shift(-7) * 100

# Seleccionar features CFTC relevantes
cftc_features = [
    'cftc_managed_net',
    'cftc_managed_net_pct',
    'cftc_managed_net_change_7d',
    'cftc_managed_net_pct_change_7d',
    'cftc_managed_producer_ratio',
    'cftc_managed_net_percentile_252d',
    'cftc_extreme_long',
    'cftc_extreme_short'
]

# Calcular correlaciones con retornos futuros
correlations = df_merged[cftc_features + ['Soybeans_return_7d_fwd']].corr()['Soybeans_return_7d_fwd'].drop('Soybeans_return_7d_fwd').sort_values(key=abs, ascending=False)

print("=" * 80)
print("CORRELACIÓN: CFTC FEATURES vs SOYBEANS RETURN (7d forward)")
print("=" * 80)
print(correlations)
print(f"\n✓ Nota: Correlación > 0.05 puede ser relevante para predicción")
print(f"  Feature más correlacionada: {correlations.index[0]} ({correlations.iloc[0]:.4f})")

# Visualizar correlaciones
fig, ax = plt.subplots(figsize=(10, 6))
correlations.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Correlation with Soybeans 7d Forward Return', fontsize=12)
ax.set_title('CFTC Features Correlation with Future Soybeans Returns', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'cftc_correlation_returns.png', dpi=300, bbox_inches='tight')
print("\n✓ Gráfico guardado: reports/figures/cftc_correlation_returns.png")
plt.show()

## 7. Guardar Dataset con CFTC Features (Step 5)

In [ ]:
# Eliminar columna auxiliar de retornos futuros (no debe usarse como feature)
df_final = df_merged.drop(columns=['Soybeans_return_7d_fwd'], errors='ignore')

# Guardar dataset Step 5
output_file = PROCESSED_DIR / 'features_step5_cftc.csv'
df_final.to_csv(output_file)

print("=" * 80)
print("DATASET GUARDADO - STEP 5 (Climate + CFTC Features)")
print("=" * 80)
print(f"Archivo: {output_file}")
print(f"Shape: {df_final.shape}")
print(f"Período: {df_final.index.min()} → {df_final.index.max()}")
print(f"Features totales: {len(df_final.columns):,}")
print(f"Features CFTC agregadas: {len(cftc_features)}")

print(f"\n✓ CFTC Features agregadas:")
for feat in df_cftc.columns:
    print(f"  - {feat}")

print("\n" + "=" * 80)
print("PRÓXIMO PASO:")
print("=" * 80)
print("Ejecutar notebook 2.5-final-dataset-preparation.ipynb para:")
print("  1. Cargar features_step5_cftc.csv")
print("  2. Limpieza final de NaNs")
print("  3. Validación de calidad")
print("  4. Guardar features_final_modeling.csv")

## Resumen

**CFTC Features agregadas al pipeline:**

1. **Posiciones base:**
   - `cftc_managed_net`: Posición neta especuladores (contratos)
   - `cftc_managed_net_pct`: Posición neta normalizada (% Open Interest)
   - `cftc_producer_net`: Posición neta hedgers
   - `cftc_open_interest`: Liquidez del mercado

2. **Momentum:**
   - `cftc_managed_net_change_7d`: Cambio semanal contratos
   - `cftc_managed_net_pct_change_7d`: Cambio semanal %
   - `cftc_open_interest_change_7d`: Cambio en liquidez

3. **Ratios:**
   - `cftc_managed_producer_ratio`: Especuladores / Hedgers

4. **Extremos (sentiment reversal signals):**
   - `cftc_managed_net_percentile_252d`: Percentil histórico (1 año)
   - `cftc_extreme_long`: Señal extremo largo (potential reversal)
   - `cftc_extreme_short`: Señal extremo corto (potential reversal)

**Interpretación clave:**
- Managed Net > 0: Sentiment alcista (especuladores largos)
- Managed Net < 0: Sentiment bajista (especuladores cortos)
- Extremos (percentil >90 o <10): Potencial reversión de tendencia
- Cambios week-over-week: Momentum del sentiment

**Período:** 2015-01-06 a 2025-10-28 (3,949 días con data CFTC)